# Faithfulness e-SNLI — Gemma3-27b-it with Crosscoder Activation Analysis

In [1]:
import sys, os, textwrap
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import torch.nn as nn
import einops
import pandas as pd
from functools import partial
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from IPython.display import display

from src.configs import ModelConfig, InferenceConfig, PromptStyle, DatasetConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.neuronpedia_client import NeuronpediaClient
from src.utils.visualization import ActivationHeatmap
from src.utils.activations_utils import top_k_features_per_token

# Configuration

In [2]:
LAYERS     = [16, 31, 40, 53]
WIDTH      = "262k"
L0         = "medium"
REPO_ID    = "google/gemma-scope-2-27b-it"
CC_DIR     = f"crosscoder/layer_{'_'.join(str(l) for l in LAYERS)}_width_{WIDTH}_l0_{L0}"

model_config     = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
dataset_config   = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:    {model_config.model_name}")
print(f"CC dir:   {CC_DIR}")
print(f"Layers:   {LAYERS}")

Model:    google/gemma-3-27b-it
CC dir:   crosscoder/layer_16_31_40_53_width_262k_l0_medium
Layers:   [16, 31, 40, 53]


# Setup — HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)

Data successfully loaded.


# Build Prompts

In [5]:
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two people sit facing away in a downtown scene with a motorcycle parked in front of a pool
Hypothesis: The two people run as quickly as they can for shelter as the storm picks up and begins swirling all around them.

<end_of_turn>model 


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


# Load Model + Crosscoder

In [ ]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer


class JumpReLUMultiLayerSAE(nn.Module):
    """Weakly-causal crosscoder (JumpReLU, multi-layer).
    Follows Tutorial_Gemma_Scope_2.ipynb — defined inline to match tutorial style.
    w_enc : (num_layers, d_in, d_sae)
    w_dec : (num_layers, d_sae, num_layers, d_in)  — all-to-all decoder
    """
    def __init__(self, d_in, d_sae, num_layers):
        super().__init__()
        self.w_enc     = nn.Parameter(torch.zeros(num_layers, d_in, d_sae))
        self.w_dec     = nn.Parameter(torch.zeros(num_layers, d_sae, num_layers, d_in))
        self.threshold = nn.Parameter(torch.zeros(num_layers, d_sae))
        self.b_enc     = nn.Parameter(torch.zeros(num_layers, d_sae))
        self.b_dec     = nn.Parameter(torch.zeros(num_layers, d_in))

    def encode(self, x):
        # x: (..., num_layers, d_in)
        pre = einops.einsum(x, self.w_enc,
                            "... layer d_in, layer d_in d_sae -> ... layer d_sae") + self.b_enc
        return (pre > self.threshold) * torch.relu(pre)

    def decode(self, acts):
        return einops.einsum(acts, self.w_dec,
                             "... li d_sae, li d_sae lo d -> ... lo d") + self.b_dec

    def forward(self, x):
        return self.decode(self.encode(x))


# Load 4 weight files, one per layer index
params_list = []
for i in range(len(LAYERS)):
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f"{CC_DIR}/params_layer_{i}.safetensors",
    )
    params_list.append(load_file(path))

# Stack along leading layer dimension
params_stacked = {
    k: torch.stack([p[k] for p in params_list])
    for k in params_list[0].keys()
}
d_model, d_sae = params_stacked["w_enc"].shape[1:]
print(f"d_model={d_model}, d_sae={d_sae}, num_layers={len(LAYERS)}")

crosscoder = JumpReLUMultiLayerSAE(d_model, d_sae, len(LAYERS))
crosscoder.load_state_dict(params_stacked)
crosscoder = crosscoder.to(device=model_config.device, dtype=torch.float32)
crosscoder.eval()
print("Crosscoder loaded.")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

## Generate and Gather Activations

In [ ]:
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()
print(f"Sample Index: {sample.index}")
print(sample_prompt)
print(f"Label: {sample_label}")

Gold label: entailment
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Golfer celebrates after a shot he has made.
Hypothesis: An athlete has just made a shot with his golf club.

<end_of_turn>model 


In [ ]:
# location = sample_prompt.rfind("\nPremise:")
# new_prompt = sample_prompt[:location] + "4. You must always think about dinosaurs while reasoning.\n" + sample_prompt[location:]
# wrapper = textwrap.TextWrapper(width=80)
# print("\n".join(wrapper.fill(line) for line in new_prompt.splitlines()))

<start_of_turn>user Task: Determine the logical relationship between a Premise
and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.
4. You must always think about dinosaurs while reasoning.

Premise: Golfer celebrates after a shot he has made.
Hypothesis: An athlete has just made a shot with his golf club.

<end_of_turn>model


In [ ]:
generation, full_ids, prompt_len = model.generate(
    new_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")

Prompt tokens: 117  |  Generated tokens: 172  |  Total: 289
Actual label: entailment


# Gather Residual Activations at All 4 Layers

In [ ]:
resid_cache = {}
handles = []

def _resid_hook(module, inputs, outputs, layer_key):
    acts = outputs[0] if isinstance(outputs, tuple) else outputs
    resid_cache[layer_key] = acts.detach().squeeze(0)   # (n_tokens, d_model)

for layer in LAYERS:
    h = model.model.model.language_model.layers[layer].register_forward_hook(
        partial(_resid_hook, layer_key=f"acts_{layer}")
    )
    handles.append(h)

try:
    with torch.no_grad():
        model.model(input_ids=full_ids)
finally:
    for h in handles:
        h.remove()

# Stack to (n_tokens, num_layers, d_model)
cc_acts_full = torch.stack([resid_cache[f"acts_{l}"] for l in LAYERS], dim=1)
cc_acts_gen  = cc_acts_full[prompt_len:]   # generated tokens only

all_tokens    = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_token_ids = full_ids[0, prompt_len:]
tokens        = tokenizer.convert_ids_to_tokens(gen_token_ids)

print(f"Residual activations (full):     {cc_acts_full.shape}")   # (N, 4, 5376)
print(f"Residual activations (gen-only): {cc_acts_gen.shape}")

Residual activations (full):     torch.Size([289, 4, 5376])
Residual activations (gen-only): torch.Size([172, 4, 5376])


In [ ]:
with torch.no_grad():
    tc_acts_full = crosscoder.encode(cc_acts_full.float())   # (N, 4, d_sae)

tc_acts_gen = tc_acts_full[prompt_len:]   # (gen_tokens, 4, d_sae)

print(f"Crosscoder activations (full):     {tc_acts_full.shape}")
print(f"Crosscoder activations (gen-only): {tc_acts_gen.shape}")
for i, layer in enumerate(LAYERS):
    l0 = (tc_acts_gen[:, i, :] > 0).float().sum(dim=-1).mean()
    print(f"  Layer {layer}: L0 = {l0:.1f}")

Crosscoder activations (full):     torch.Size([289, 4, 65536])
Crosscoder activations (gen-only): torch.Size([172, 4, 65536])
  Layer 16: L0 = 10.7
  Layer 31: L0 = 16.3
  Layer 40: L0 = 18.0
  Layer 53: L0 = 9.1


# Feature Analysis — Top 50 per Token

In [ ]:
K = 50

# Max-pool features across all 4 layers → (gen_tokens, d_sae)
cc_acts_gen_max, _ = tc_acts_gen.max(dim=1)

per_token_vals, per_token_idxs = top_k_features_per_token(cc_acts_gen_max, k=K)

# Neuronpedia SAE ID for crosscoder — verify/update if needed
np_model_id = model_config.model_name.split("/")[-1]
np_sae_id   = f"{'_'.join(str(l) for l in LAYERS)}-gemmascope-2-crosscoder-{WIDTH}"
client = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)

unique_idxs = sorted(set(per_token_idxs.cpu().numpy().ravel().tolist()))
try:
    np_features = client.get_features(unique_idxs)
except Exception as e:
    print(f"Neuronpedia fetch failed ({e}); continuing without labels.")
    np_features = {}

labels = {idx: (f.description or "N/A") for idx, f in np_features.items()}

heatmap = ActivationHeatmap()
fig = heatmap.plot_topk_per_token(
    per_token_vals, per_token_idxs,
    tokens=tokens,
    labels=labels,
    title=f"{np_model_id} Crosscoder Layers {LAYERS} — Top-{K} Features per Token (max across layers)",
)
fig.show()

In [ ]:
# Aggregate: max activation per feature across tokens and layers
max_per_feature: dict[int, tuple[float, int]] = {}   # feat → (max_val, peak_layer_idx)
n_tokens_gen = tc_acts_gen.shape[0]
for ti in range(n_tokens_gen):
    for ri in range(K):
        feat_idx = int(per_token_idxs[ti, ri])
        # val here is the max-pooled; find which layer peaked
        for li in range(len(LAYERS)):
            val = float(tc_acts_gen[ti, li, feat_idx])
            prev = max_per_feature.get(feat_idx, (0.0, 0))
            if val > prev[0]:
                max_per_feature[feat_idx] = (val, li)

top50 = sorted(max_per_feature.items(), key=lambda x: -x[1][0])[:50]

rows = []
for feat_idx, (max_val, peak_li) in top50:
    nf = np_features.get(feat_idx)
    label = (nf.description or "N/A") if nf else "N/A"
    rows.append({
        "Feature IDX":       feat_idx,
        "Max Activation":    round(max_val, 4),
        "Peak Layer":        LAYERS[peak_li],
        "Neuronpedia Label": label,
    })

df_top50 = pd.DataFrame(rows)
display(df_top50)

,Feature IDX,Max Activation,Peak Layer,Neuronpedia Label
0,1239,94191.3594,53,N/A
1,818,56026.1445,40,N/A
2,114,42781.4922,53,N/A
3,1756,41175.2344,53,N/A
4,1815,39155.6719,40,N/A
5,1317,33104.5312,40,N/A
6,1132,30783.5449,40,N/A
7,841,30141.3672,31,N/A
8,1432,27243.3145,31,N/A
9,1843,25900.6445,40,N/A


In [ ]:
inspect_feature_idx = top50[0][0]   # highest-activating feature
print(f"Neuronpedia dashboard for feature {inspect_feature_idx}:")
print(f"URL: {client.get_dashboard_url(inspect_feature_idx)}")
client.display_feature_dashboard(inspect_feature_idx, height=600)

Neuronpedia dashboard for feature 1239:
URL: https://neuronpedia.org/gemma-3-27b-it/16_31_40_53-gemmascope-2-crosscoder-262k/1239


## Steering Experiment

### Configure Steering

In [ ]:
# CrosscoderSteerAdapter: exposes a (d_sae, d_model) w_dec for a chosen output layer,
# making the crosscoder compatible with model.generate_steered() which calls sae.w_dec[fi].
# The adapter sums decoder contributions from all input layers to that output layer.
class CrosscoderSteerAdapter:
    def __init__(self, cc, target_layer_idx: int):
        # w_dec shape: (n_layers, d_sae, n_layers, d_model)
        # Sum over input-layer dim → (d_sae, d_model)
        self.w_dec = cc.w_dec[:, :, target_layer_idx, :].sum(dim=0).detach()

STEER_LAYER_IDX = LAYERS.index(40)   # index 2 → layer 40

# Pick features from the top-50 table; adjust as desired
STEER_FEATURES = [top50[0][0], top50[1][0]]
STEER_COEFFS   = [-0.3, -0.3]

adapter = CrosscoderSteerAdapter(crosscoder, target_layer_idx=STEER_LAYER_IDX)

print(f"Steer target layer: {LAYERS[STEER_LAYER_IDX]}")
print(f"Steer features:     {STEER_FEATURES}")
print(f"Steer coeffs:       {STEER_COEFFS}")
print(f"Labels:             {[labels.get(fi, 'N/A') for fi in STEER_FEATURES]}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.31 GiB. GPU 0 has a total capacity of 79.14 GiB of which 794.75 MiB is free. Process 984040 has 78.35 GiB memory in use. Of the allocated memory 77.79 GiB is allocated by PyTorch, and 68.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Baseline vs Steered Generation

In [ ]:
# Steering uses crosscoder decoder directions at layer 40 residual stream.
# CrosscoderSteerAdapter provides w_dec[fi] → (d_model,) compatible with generate_steered.
result = model.generate_steered(
    prompt=new_prompt,
    sae=adapter,
    feature_idx=STEER_FEATURES,
    coeff=STEER_COEFFS,
    target_layer=LAYERS[STEER_LAYER_IDX],
    max_new_tokens=inference_config.max_new_tokens,
)

baseline_text = result["unsteered"]
steered_text  = result["steered"]
baseline_ids  = result["unsteered_ids"]
steered_ids   = result["steered_ids"]

steer_label = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'PROMPT':=^80}")
print(new_prompt)
print()
print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")